In [130]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import MNIST

In [131]:
#dataset and dataloaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

In [132]:
trainset = MNIST(root= "./data3", train= True, download = True, transform=transform)
testset = MNIST(root= "./data3", train= False, download = True, transform=transform)

In [133]:
trainloader = DataLoader(trainset, batch_size=64, shuffle = True)
testloader = DataLoader(testset, batch_size=64)

## Build the CNN

In [134]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers= nn.Sequential(
            nn.Linear(7*7*64, 128),
            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x

In [135]:
model = CNN()

In [136]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [137]:
epochs = 5
for epoch in range (epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        outputs = model.forward(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")

epoch = 1/5 & loss = 0.15739391997867602
epoch = 2/5 & loss = 0.045191151101508543
epoch = 3/5 & loss = 0.030824670580395463
epoch = 4/5 & loss = 0.02314353802987039
epoch = 5/5 & loss = 0.016234799829248185


In [138]:
correct_labels=0
tot_labels=0

model.eval()
with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        tot_labels += labels.size(0)

    print(f"Accuracy for CNN: {correct_labels/tot_labels * 100}")

Accuracy for CNN: 99.05000000000001


## RNN

In [139]:
class RNN(nn.Module):

    def __init__(self):

        super(RNN, self).__init__()

        self.hidden_size = 128

        self.rnn = nn.RNN(input_size=28,hidden_size=128,num_layers=1,batch_first=True)

        self.fc = nn.Linear(128, 10)

    def forward(self, x):

        x = x.squeeze(1)

        h0 = torch.zeros(1, x.size(0), self.hidden_size).to(x.device)

        out, _ = self.rnn(x, h0)

        out = self.fc(out[:, -1, :])

        return out

In [140]:
model = RNN()

In [141]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [142]:
epochs = 5

for epoch in range(epochs):

    running_loss = 0.0

    for images, labels in trainloader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}")

Epoch [1/5], Loss: 757.4861
Epoch [2/5], Loss: 319.6610
Epoch [3/5], Loss: 216.4616
Epoch [4/5], Loss: 186.6904
Epoch [5/5], Loss: 166.9620


In [143]:
correct_vals = 0
total_vals = 0

model.eval()

with torch.no_grad():

    for images, labels in testloader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total_vals += labels.size(0)

        correct_vals += (predicted == labels).sum().item()

accuracy = 100 * correct_vals / total_vals

print(f"Accuracy for RNN = {accuracy:.2f}%")

Accuracy for RNN = 95.98%
